# Análisis reproducible del estudio de accesibilidad web en Europa

Este cuaderno contiene el análisis estadístico reproducible de un estudio sobre la accesibilidad web de servicios digitales de interés público en Europa.

El análisis parte del archivo `uais_analysis_data2026.xlsx`, que contiene:

- Información general de los países e indicadores nacionales.
- Resultados de accesibilidad obtenidos mediante ROAW.
- Resultados de accesibilidad obtenidos mediante Lighthouse.
- Metadatos y fuentes de los indicadores utilizados.

ROAW y Lighthouse se analizan como medidas independientes de accesibilidad. Sus puntuaciones se mantienen en las escalas originales de cada herramienta y no se combinan en una puntuación única.

Los resultados derivados se calculan mediante código a partir de los datos originales, evitando cálculos manuales intermedios.

## 1. Carga y validación del conjunto de datos

Se carga el archivo de datos utilizado en el estudio y se comprueba que contiene las cuatro hojas previstas.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving uais_analysis_data_2026.xlsx to uais_analysis_data_2026.xlsx


In [ ]:
import pandas as pd
import numpy as np

archivo = "uais_analysis_data_2026.xlsx"

xls = pd.ExcelFile(archivo)

print("Hojas disponibles:")
print(xls.sheet_names)

Hojas disponibles:
['country_data', 'ROAW', 'Lighthouse', 'indicators_metadata']


### 1.1. Comprobación de la estructura

Se cargan las cuatro hojas del archivo y se verifica:

- El número de países.
- El número de evaluaciones realizadas con cada herramienta.
- El número de indicadores.
- La ausencia de registros duplicados por país y categoría.

In [ ]:
country = pd.read_excel(archivo, sheet_name="country_data")
roaw = pd.read_excel(archivo, sheet_name="ROAW")
lighthouse = pd.read_excel(archivo, sheet_name="Lighthouse")
metadata = pd.read_excel(archivo, sheet_name="indicators_metadata")

print("country_data:", country.shape)
print("ROAW:", roaw.shape)
print("Lighthouse:", lighthouse.shape)
print("indicators_metadata:", metadata.shape)

print("\nDuplicados Country en country_data:",
      country["Country"].duplicated().sum())

print("Duplicados Country + Category en ROAW:",
      roaw.duplicated(subset=["Country", "Category"]).sum())

print("Duplicados Country + Category en Lighthouse:",
      lighthouse.duplicated(subset=["Country", "Category"]).sum())

country_data: (31, 7)
ROAW: (155, 5)
Lighthouse: (155, 5)
indicators_metadata: (4, 5)

Duplicados Country en country_data: 0
Duplicados Country + Category en ROAW: 0
Duplicados Country + Category en Lighthouse: 0


### 1.2. Validación de tipos de datos y valores ausentes

Se comprueban los tipos de datos, la presencia de valores ausentes y la distribución de las evaluaciones entre las cinco categorías de sitios web.

Los valores ausentes se mantienen como tales y no se sustituyen ni estiman.

In [ ]:
print("TIPOS DE DATOS\n")

print("country_data:")
print(country.dtypes)

print("\nROAW:")
print(roaw.dtypes)

print("\nLighthouse:")
print(lighthouse.dtypes)


print("\n\nVALORES AUSENTES\n")

print("country_data:")
print(country.isna().sum())

print("\nROAW:")
print(roaw.isna().sum())

print("\nLighthouse:")
print(lighthouse.isna().sum())


print("\n\nCATEGORÍAS\n")

print("ROAW:")
print(roaw["Category"].value_counts())

print("\nLighthouse:")
print(lighthouse["Category"].value_counts())

TIPOS DE DATOS

country_data:
Country              object
Regulatory group     object
M49 region           object
HDI                 float64
SPI                 float64
EGDI                float64
Internet access     float64
dtype: object

ROAW:
Country                    object
Category                   object
URL                        object
Score                     float64
Evaluation date    datetime64[ns]
dtype: object

Lighthouse:
Country                    object
Category                   object
URL                        object
Score                     float64
Evaluation date    datetime64[ns]
dtype: object


VALORES AUSENTES

country_data:
Country             0
Regulatory group    0
M49 region          0
HDI                 0
SPI                 1
EGDI                1
Internet access     0
dtype: int64

ROAW:
Country             0
Category            0
URL                 0
Score              28
Evaluation date    28
dtype: int64

Lighthouse:
Country            0
Categor

### 1.3. Validación de las evaluaciones de accesibilidad

Se comprueba que las puntuaciones se encuentran dentro de las escalas originales de cada herramienta:

- ROAW: 0–10.
- Lighthouse: 0–100.

Las puntuaciones de ambas herramientas se mantienen en sus escalas originales y se analizan de forma independiente.

Posteriormente se vinculan las evaluaciones utilizando el país y la categoría del sitio web como identificadores. Se comprueba que las URL correspondientes coincidan entre ambas fuentes y se examina la proximidad temporal entre las fechas de evaluación.

Para cada sitio web se registra también el número de herramientas que proporcionaron una puntuación válida.

In [ ]:
print("RANGO DE PUNTUACIONES\n")

print("ROAW:")
print("Mínimo:", roaw["Score"].min())
print("Máximo:", roaw["Score"].max())

print("\nLighthouse:")
print("Mínimo:", lighthouse["Score"].min())
print("Máximo:", lighthouse["Score"].max())


# Unimos las dos herramientas por país y categoría
accessibility = roaw.merge(
    lighthouse,
    on=["Country", "Category"],
    how="outer",
    suffixes=("_ROAW", "_Lighthouse"),
    validate="one_to_one"
)


# Comprobamos que las URLs coinciden
accessibility["URL_match"] = (
    accessibility["URL_ROAW"] == accessibility["URL_Lighthouse"]
)

print("\nURLs diferentes entre herramientas:",
      (~accessibility["URL_match"]).sum())


# Comparamos las fechas de evaluación
accessibility["Evaluation date_ROAW"] = pd.to_datetime(
    accessibility["Evaluation date_ROAW"]
)

accessibility["Evaluation date_Lighthouse"] = pd.to_datetime(
    accessibility["Evaluation date_Lighthouse"]
)

accessibility["Date_difference_days"] = (
    accessibility["Evaluation date_ROAW"]
    - accessibility["Evaluation date_Lighthouse"]
).dt.days.abs()

print("\nEvaluaciones realizadas el mismo día:",
      (accessibility["Date_difference_days"] == 0).sum())

print("Evaluaciones realizadas en fechas diferentes:",
      (accessibility["Date_difference_days"] > 0).sum())

print("Máxima diferencia entre evaluaciones:",
      accessibility["Date_difference_days"].max(), "días")


# Contamos cuántas herramientas proporcionaron puntuación
accessibility["n_tools"] = accessibility[
    ["Score_ROAW", "Score_Lighthouse"]
].notna().sum(axis=1)

print("\nNúmero de webs según herramientas disponibles:")
print(accessibility["n_tools"].value_counts().sort_index())


print("\nWebs sin puntuación en ninguna herramienta:")
print(
    accessibility.loc[
        accessibility["n_tools"] == 0,
        ["Country", "Category"]
    ].to_string(index=False)
)

RANGO DE PUNTUACIONES

ROAW:
Mínimo: 3.2
Máximo: 9.5

Lighthouse:
Mínimo: 34.0
Máximo: 100.0

URLs diferentes entre herramientas: 0

Evaluaciones realizadas el mismo día: 126
Evaluaciones realizadas en fechas diferentes: 0
Máxima diferencia entre evaluaciones: 0.0 días

Número de webs según herramientas disponibles:
n_tools
1     29
2    126
Name: count, dtype: int64

Webs sin puntuación en ninguna herramienta:
Empty DataFrame
Columns: [Country, Category]
Index: []


## 2. Construcción del conjunto de datos a nivel de país

Las puntuaciones de ROAW y Lighthouse se reorganizan de forma independiente para obtener, para cada herramienta, una fila por país y una columna por categoría de sitio web.

Para cada país y herramienta se calculan:

- El número de categorías con una puntuación válida.
- La puntuación media de accesibilidad a partir de las categorías disponibles.

Las puntuaciones de ROAW y Lighthouse se mantienen en sus escalas originales y no se combinan entre sí.

Para considerar válida la puntuación media nacional se exige disponer de resultados en al menos tres de las cinco categorías analizadas. Este criterio garantiza que la medida represente a la mayoría de las categorías incluidas en el estudio y evita calcular una puntuación nacional a partir de una cobertura excesivamente reducida.

El número de categorías disponibles se conserva para evaluar la cobertura de cada herramienta y realizar posteriormente análisis de sensibilidad restringidos a los países con información completa en las cinco categorías.

In [ ]:
# Orden de las cinco categorías analizadas
category_order = [
    "National gov.",
    "Local gov.",
    "Tourism",
    "Public transport",
    "News portals"
]


# ============================================================
# ROAW
# ============================================================

roaw_by_country = (
    accessibility
    .pivot(
        index="Country",
        columns="Category",
        values="Score_ROAW"
    )
    .reindex(columns=category_order)
)

# Número de categorías disponibles
roaw_by_country["n_categories_ROAW"] = (
    roaw_by_country[category_order]
    .notna()
    .sum(axis=1)
)

# Media de las categorías disponibles
roaw_by_country["Mean_ROAW"] = (
    roaw_by_country[category_order]
    .mean(axis=1, skipna=True)
)

# Se exige un mínimo de 3 categorías para considerar válida
# la puntuación media nacional
roaw_by_country.loc[
    roaw_by_country["n_categories_ROAW"] < 3,
    "Mean_ROAW"
] = np.nan



# ============================================================
# LIGHTHOUSE
# ============================================================

lighthouse_by_country = (
    accessibility
    .pivot(
        index="Country",
        columns="Category",
        values="Score_Lighthouse"
    )
    .reindex(columns=category_order)
)

# Número de categorías disponibles
lighthouse_by_country["n_categories_Lighthouse"] = (
    lighthouse_by_country[category_order]
    .notna()
    .sum(axis=1)
)

# Media de las categorías disponibles
lighthouse_by_country["Mean_Lighthouse"] = (
    lighthouse_by_country[category_order]
    .mean(axis=1, skipna=True)
)

# Se exige un mínimo de 3 categorías
lighthouse_by_country.loc[
    lighthouse_by_country["n_categories_Lighthouse"] < 3,
    "Mean_Lighthouse"
] = np.nan



# Extraer las variables resumen de cada herramienta
roaw_summary = (
    roaw_by_country[
        [
            "n_categories_ROAW",
            "Mean_ROAW"
        ]
    ]
    .reset_index()
)

lighthouse_summary = (
    lighthouse_by_country[
        [
            "n_categories_Lighthouse",
            "Mean_Lighthouse"
        ]
    ]
    .reset_index()
)


# Incorporar las puntuaciones a los indicadores nacionales
analysis_data = (
    country
    .merge(
        roaw_summary,
        on="Country",
        how="left",
        validate="one_to_one"
    )
    .merge(
        lighthouse_summary,
        on="Country",
        how="left",
        validate="one_to_one"
    )
)


print("Dimensiones del dataset analítico:")
print(analysis_data.shape)


print("\nCobertura ROAW:")
print(
    analysis_data["n_categories_ROAW"]
    .value_counts()
    .sort_index()
)


print("\nCobertura Lighthouse:")
print(
    analysis_data["n_categories_Lighthouse"]
    .value_counts()
    .sort_index()
)


print("\nPaíses sin puntuación media nacional válida en ROAW:")
print(
    analysis_data.loc[
        analysis_data["Mean_ROAW"].isna(),
        ["Country", "n_categories_ROAW"]
    ].to_string(index=False)
)


print("\nPaíses sin puntuación media nacional válida en Lighthouse:")
print(
    analysis_data.loc[
        analysis_data["Mean_Lighthouse"].isna(),
        ["Country", "n_categories_Lighthouse"]
    ].to_string(index=False)
)



print("\nPrimeras filas del dataset analítico:")
print(
    analysis_data[
        [
            "Country",
            "n_categories_ROAW",
            "Mean_ROAW",
            "n_categories_Lighthouse",
            "Mean_Lighthouse",
            "HDI",
            "SPI",
            "EGDI",
            "Internet access"
        ]
    ].head().to_string(index=False)
)

Dimensiones del dataset analítico:
(31, 11)

Cobertura ROAW:
n_categories_ROAW
2     1
3     9
4     7
5    14
Name: count, dtype: int64

Cobertura Lighthouse:
n_categories_Lighthouse
4     1
5    30
Name: count, dtype: int64

Países sin puntuación media nacional válida en ROAW:
Country  n_categories_ROAW
 Greece                  2

Países sin puntuación media nacional válida en Lighthouse:
Empty DataFrame
Columns: [Country, n_categories_Lighthouse]
Index: []

Primeras filas del dataset analítico:
 Country  n_categories_ROAW  Mean_ROAW  n_categories_Lighthouse  Mean_Lighthouse   HDI   SPI    EGDI  Internet access
 Austria                  5       6.26                        5            96.20 0.930 87.80 0.90653         0.949197
 Belgium                  3       5.70                        5            93.60 0.951 87.51 0.81212         0.957758
Bulgaria                  5       5.84                        5            81.60 0.845 75.73 0.81453         0.824390
 Croatia                 

### 2.1. Tratamiento de los datos de accesibilidad ausentes

Los análisis se realizan utilizando los casos válidos disponibles, sin imputación de puntuaciones ausentes.

Para los análisis a nivel de país se considera válida una puntuación media nacional cuando existen resultados para al menos tres de las cinco categorías de sitios web. El número de categorías disponibles se conserva como indicador de cobertura.

Como análisis de sensibilidad, las asociaciones principales se repetirán posteriormente utilizando únicamente los países con resultados válidos en las cinco categorías de cada herramienta.

## 3. Estadísticos descriptivos

Se calculan estadísticos descriptivos de las puntuaciones de accesibilidad para cada una de las cinco categorías de sitios web y para cada herramienta de evaluación de forma independiente.

Para cada combinación de herramienta y categoría se obtiene:

- número de observaciones válidas (`n`);
- media;
- desviación estándar;
- mediana;
- mínimo;
- máximo.

Las puntuaciones de ROAW y Lighthouse se mantienen en sus escalas originales (0–10 y 0–100, respectivamente). Por este motivo, los valores absolutos de ambas herramientas no deben interpretarse como directamente equivalentes.

In [ ]:
# ============================================================
# ESTADÍSTICOS DESCRIPTIVOS POR CATEGORÍA Y HERRAMIENTA
# ============================================================

descriptive_rows = []

for tool, score_col in [
    ("ROAW", "Score_ROAW"),
    ("Lighthouse", "Score_Lighthouse")
]:
    for category in category_order:

        values = accessibility.loc[
            accessibility["Category"] == category,
            score_col
        ].dropna()

        descriptive_rows.append({
            "Tool": tool,
            "Category": category,
            "n": len(values),
            "Mean": values.mean(),
            "SD": values.std(ddof=1),
            "Median": values.median(),
            "Min": values.min(),
            "Max": values.max()
        })


descriptives_category = pd.DataFrame(descriptive_rows)


# Ordenar categorías y herramientas
descriptives_category["Category"] = pd.Categorical(
    descriptives_category["Category"],
    categories=category_order,
    ordered=True
)

descriptives_category["Tool"] = pd.Categorical(
    descriptives_category["Tool"],
    categories=["ROAW", "Lighthouse"],
    ordered=True
)

descriptives_category = (
    descriptives_category
    .sort_values(["Tool", "Category"])
    .reset_index(drop=True)
)


# Versión redondeada para visualización
descriptives_category_display = descriptives_category.copy()

for col in ["Mean", "SD", "Median", "Min", "Max"]:
    descriptives_category_display[col] = (
        descriptives_category_display[col].round(2)
    )


display(descriptives_category_display)

,Tool,Category,n,Mean,SD,Median,Min,Max
0,ROAW,National gov.,22,7.23,1.32,6.80,5.3,9.5
1,ROAW,Local gov.,22,6.59,1.46,6.50,3.5,9.5
2,ROAW,Tourism,28,6.09,1.22,5.80,3.7,9.4
3,ROAW,Public transport,26,5.86,1.54,6.05,3.2,8.4
4,ROAW,News portals,29,6.06,1.08,6.10,3.9,8.4
5,Lighthouse,National gov.,31,92.23,13.93,96.00,34.0,100.0
6,Lighthouse,Local gov.,31,92.29,10.32,97.00,63.0,100.0
7,Lighthouse,Tourism,31,89.97,8.45,92.00,73.0,100.0
8,Lighthouse,Public transport,30,90.17,8.89,92.50,69.0,100.0
9,Lighthouse,News portals,31,88.77,9.14,90.00,65.0,100.0


### 3.1. Estadísticos descriptivos de las puntuaciones medias por país

Se resumen también las puntuaciones medias nacionales utilizadas posteriormente en los análisis de asociación con los indicadores nacionales.

La media de ROAW se calcula únicamente para los países con al menos tres categorías disponibles, mientras que Lighthouse dispone de cobertura suficiente para los 31 países analizados.

In [ ]:
country_mean_descriptives = pd.DataFrame([
    {
        "Tool": "ROAW",
        "n": analysis_data["Mean_ROAW"].notna().sum(),
        "Mean": analysis_data["Mean_ROAW"].mean(),
        "SD": analysis_data["Mean_ROAW"].std(ddof=1),
        "Median": analysis_data["Mean_ROAW"].median(),
        "Min": analysis_data["Mean_ROAW"].min(),
        "Max": analysis_data["Mean_ROAW"].max()
    },
    {
        "Tool": "Lighthouse",
        "n": analysis_data["Mean_Lighthouse"].notna().sum(),
        "Mean": analysis_data["Mean_Lighthouse"].mean(),
        "SD": analysis_data["Mean_Lighthouse"].std(ddof=1),
        "Median": analysis_data["Mean_Lighthouse"].median(),
        "Min": analysis_data["Mean_Lighthouse"].min(),
        "Max": analysis_data["Mean_Lighthouse"].max()
    }
])


country_mean_descriptives_display = country_mean_descriptives.copy()

for col in ["Mean", "SD", "Median", "Min", "Max"]:
    country_mean_descriptives_display[col] = (
        country_mean_descriptives_display[col].round(2)
    )


display(country_mean_descriptives_display)

,Tool,n,Mean,SD,Median,Min,Max
0,ROAW,30,6.31,0.77,6.26,4.67,7.87
1,Lighthouse,31,90.65,6.27,91.40,73.00,99.40


## 4. Asociación entre accesibilidad e indicadores nacionales

Se analiza la asociación entre las puntuaciones medias nacionales de accesibilidad obtenidas mediante ROAW y Lighthouse y cuatro indicadores nacionales:

- Human Development Index (HDI).
- Social Progress Index (SPI).
- E-Government Development Index (EGDI).
- Acceso a Internet.

Para cada combinación de herramienta e indicador se calculan los coeficientes de correlación de Pearson y Spearman utilizando los casos válidos disponibles.

Pearson permite evaluar asociaciones lineales entre las variables, mientras que Spearman proporciona una medida complementaria basada en rangos y menos sensible a distribuciones no normales y valores extremos.

Dado que se realizan ocho contrastes para cada método de correlación (cuatro indicadores por dos herramientas), los valores p se ajustan mediante el procedimiento de Benjamini-Hochberg para controlar la tasa de falsos descubrimientos (FDR). La corrección se aplica de forma independiente a las pruebas de Pearson y Spearman.

In [ ]:
from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests


indicators = [
    "HDI",
    "SPI",
    "EGDI",
    "Internet access"
]

tools = {
    "ROAW": "Mean_ROAW",
    "Lighthouse": "Mean_Lighthouse"
}


correlation_rows = []

for tool, score_col in tools.items():

    for indicator in indicators:

        data = analysis_data[
            [score_col, indicator]
        ].dropna()

        n = len(data)

        pearson_r, pearson_p = pearsonr(
            data[score_col],
            data[indicator]
        )

        spearman_rho, spearman_p = spearmanr(
            data[score_col],
            data[indicator]
        )

        correlation_rows.append({
            "Tool": tool,
            "Indicator": indicator,
            "n": n,
            "Pearson_r": pearson_r,
            "Pearson_p": pearson_p,
            "Spearman_rho": spearman_rho,
            "Spearman_p": spearman_p
        })


correlations_main = pd.DataFrame(correlation_rows)


# Corrección FDR por separado para Pearson y Spearman
correlations_main["Pearson_q"] = multipletests(
    correlations_main["Pearson_p"],
    method="fdr_bh"
)[1]

correlations_main["Spearman_q"] = multipletests(
    correlations_main["Spearman_p"],
    method="fdr_bh"
)[1]


# Versión redondeada para visualización
correlations_main_display = correlations_main.copy()

for col in [
    "Pearson_r",
    "Pearson_p",
    "Pearson_q",
    "Spearman_rho",
    "Spearman_p",
    "Spearman_q"
]:
    correlations_main_display[col] = (
        correlations_main_display[col].round(4)
    )


display(correlations_main_display)

,Tool,Indicator,n,Pearson_r,Pearson_p,Spearman_rho,Spearman_p,Pearson_q,Spearman_q
0,ROAW,HDI,30,0.5257,0.0028,0.5086,0.0041,0.0046,0.0066
1,ROAW,SPI,29,0.5442,0.0023,0.5439,0.0023,0.0045,0.0046
2,ROAW,EGDI,29,0.4686,0.0104,0.4029,0.0302,0.0118,0.0345
3,ROAW,Internet access,30,0.4116,0.0238,0.3557,0.0537,0.0238,0.0537
4,Lighthouse,HDI,31,0.7419,0.0000,0.6847,0.0000,0.0000,0.0001
5,Lighthouse,SPI,30,0.8027,0.0000,0.7422,0.0000,0.0000,0.0000
6,Lighthouse,EGDI,30,0.6032,0.0004,0.5608,0.0013,0.0011,0.0034
7,Lighthouse,Internet access,31,0.4888,0.0053,0.4229,0.0178,0.0070,0.0237


## 5. Asociación con indicadores nacionales por categoría de sitio web

Como análisis exploratorio, se examina si las asociaciones observadas a nivel nacional se reproducen dentro de cada una de las cinco categorías de sitios web.

Para cada combinación de herramienta, categoría e indicador nacional se calculan los coeficientes de correlación de Pearson y Spearman utilizando los casos válidos disponibles.

En total se realizan 40 contrastes por método de correlación:

- 2 herramientas;
- 5 categorías de sitios web;
- 4 indicadores nacionales.

Dado el elevado número de comparaciones, los valores p se ajustan mediante el procedimiento de Benjamini-Hochberg para controlar la tasa de falsos descubrimientos (FDR). La corrección se aplica de forma independiente a las pruebas de Pearson y Spearman.

Estos análisis se consideran exploratorios y se interpretan atendiendo tanto a la magnitud y dirección de los coeficientes como a los valores ajustados.

In [ ]:
# Incorporar los indicadores nacionales a los datos por sitio web
category_data = accessibility.merge(
    country[
        [
            "Country",
            "HDI",
            "SPI",
            "EGDI",
            "Internet access"
        ]
    ],
    on="Country",
    how="left",
    validate="many_to_one"
)


category_correlation_rows = []

for tool, score_col in {
    "ROAW": "Score_ROAW",
    "Lighthouse": "Score_Lighthouse"
}.items():

    for category in category_order:

        for indicator in indicators:

            data = category_data.loc[
                category_data["Category"] == category,
                [score_col, indicator]
            ].dropna()

            n = len(data)

            pearson_r, pearson_p = pearsonr(
                data[score_col],
                data[indicator]
            )

            spearman_rho, spearman_p = spearmanr(
                data[score_col],
                data[indicator]
            )

            category_correlation_rows.append({
                "Tool": tool,
                "Category": category,
                "Indicator": indicator,
                "n": n,
                "Pearson_r": pearson_r,
                "Pearson_p": pearson_p,
                "Spearman_rho": spearman_rho,
                "Spearman_p": spearman_p
            })


category_correlations = pd.DataFrame(
    category_correlation_rows
)


# Corrección FDR sobre los 40 contrastes de cada método
category_correlations["Pearson_q"] = multipletests(
    category_correlations["Pearson_p"],
    method="fdr_bh"
)[1]

category_correlations["Spearman_q"] = multipletests(
    category_correlations["Spearman_p"],
    method="fdr_bh"
)[1]


# Versión redondeada para visualización
category_correlations_display = (
    category_correlations.copy()
)

for col in [
    "Pearson_r",
    "Pearson_p",
    "Pearson_q",
    "Spearman_rho",
    "Spearman_p",
    "Spearman_q"
]:
    category_correlations_display[col] = (
        category_correlations_display[col].round(4)
    )


display(category_correlations_display)

,Tool,Category,Indicator,n,Pearson_r,Pearson_p,Spearman_rho,Spearman_p,Pearson_q,Spearman_q
0,ROAW,National gov.,HDI,22,0.3655,0.0944,0.3793,0.0817,0.1642,0.2043
1,ROAW,National gov.,SPI,21,0.3436,0.1272,0.3610,0.1079,0.2036,0.2263
2,ROAW,National gov.,EGDI,21,0.3946,0.0767,0.3637,0.1051,0.1534,0.2263
3,ROAW,National gov.,Internet access,22,0.3703,0.0898,0.3222,0.1437,0.1632,0.2593
4,ROAW,Local gov.,HDI,22,0.4809,0.0235,0.5486,0.0082,0.0723,0.0410
5,ROAW,Local gov.,SPI,21,0.5835,0.0055,0.6093,0.0034,0.0244,0.0213
6,ROAW,Local gov.,EGDI,21,0.4275,0.0532,0.3683,0.1005,0.1183,0.2263
7,ROAW,Local gov.,Internet access,22,0.4324,0.0444,0.4329,0.0442,0.1183,0.1359
8,ROAW,Tourism,HDI,28,0.1562,0.4274,0.2363,0.2261,0.4885,0.2918
9,ROAW,Tourism,SPI,28,0.1708,0.3849,0.2389,0.2209,0.4615,0.2918


In [ ]:
print("Resultados con q < .05 en Pearson:")
display(
    category_correlations_display.loc[
        category_correlations["Pearson_q"] < 0.05
    ]
)

print("\nResultados con q < .05 en Spearman:")
display(
    category_correlations_display.loc[
        category_correlations["Spearman_q"] < 0.05
    ]
)

Resultados con q < .05 en Pearson:


,Tool,Category,Indicator,n,Pearson_r,Pearson_p,Spearman_rho,Spearman_p,Pearson_q,Spearman_q
5,ROAW,Local gov.,SPI,21,0.5835,0.0055,0.6093,0.0034,0.0244,0.0213
20,Lighthouse,National gov.,HDI,31,0.4999,0.0042,0.2126,0.2508,0.0210,0.3039
21,Lighthouse,National gov.,SPI,30,0.5292,0.0026,0.2562,0.1717,0.0151,0.2748
22,Lighthouse,National gov.,EGDI,30,0.4618,0.0102,0.2379,0.2054,0.0371,0.2918
24,Lighthouse,Local gov.,HDI,31,0.4814,0.0061,0.4572,0.0097,0.0244,0.0431
25,Lighthouse,Local gov.,SPI,30,0.5322,0.0025,0.5348,0.0023,0.0151,0.0186
28,Lighthouse,Tourism,HDI,31,0.6011,0.0003,0.6524,0.0001,0.0051,0.0014
29,Lighthouse,Tourism,SPI,30,0.6844,0.0000,0.6784,0.0000,0.0012,0.0014
30,Lighthouse,Tourism,EGDI,30,0.5720,0.0010,0.6015,0.0004,0.0096,0.0044
31,Lighthouse,Tourism,Internet access,31,0.4435,0.0125,0.4061,0.0234,0.0415,0.0851



Resultados con q < .05 en Spearman:


,Tool,Category,Indicator,n,Pearson_r,Pearson_p,Spearman_rho,Spearman_p,Pearson_q,Spearman_q
4,ROAW,Local gov.,HDI,22,0.4809,0.0235,0.5486,0.0082,0.0723,0.0410
5,ROAW,Local gov.,SPI,21,0.5835,0.0055,0.6093,0.0034,0.0244,0.0213
24,Lighthouse,Local gov.,HDI,31,0.4814,0.0061,0.4572,0.0097,0.0244,0.0431
25,Lighthouse,Local gov.,SPI,30,0.5322,0.0025,0.5348,0.0023,0.0151,0.0186
28,Lighthouse,Tourism,HDI,31,0.6011,0.0003,0.6524,0.0001,0.0051,0.0014
29,Lighthouse,Tourism,SPI,30,0.6844,0.0000,0.6784,0.0000,0.0012,0.0014
30,Lighthouse,Tourism,EGDI,30,0.5720,0.0010,0.6015,0.0004,0.0096,0.0044
36,Lighthouse,News portals,HDI,31,0.5977,0.0004,0.6114,0.0003,0.0051,0.0034
37,Lighthouse,News portals,SPI,30,0.5383,0.0021,0.5131,0.0037,0.0151,0.0213


## 6. Diferencias regionales en accesibilidad

Se analiza si las puntuaciones de accesibilidad difieren entre las cuatro subregiones europeas definidas por la clasificación M49 de Naciones Unidas:

- Eastern Europe.
- Northern Europe.
- Southern Europe.
- Western Europe.

Chipre se excluye únicamente de los análisis regionales, ya que la clasificación M49 de Naciones Unidas lo sitúa en Western Asia.

El análisis se realiza por separado para ROAW y Lighthouse. Para cada herramienta se calculan estadísticos descriptivos por región y se aplica la prueba no paramétrica de Kruskal-Wallis para evaluar la existencia de diferencias entre las cuatro regiones.

Como medida del tamaño del efecto se calcula epsilon cuadrado (`ε²`).

In [ ]:
from scipy.stats import kruskal


european_regions = [
    "Eastern Europe",
    "Northern Europe",
    "Southern Europe",
    "Western Europe"
]


# Excluir países no clasificados en las cuatro subregiones europeas
regional_data = analysis_data[
    analysis_data["M49 region"].isin(european_regions)
].copy()


print("Número de países por región:")
print(
    regional_data["M49 region"]
    .value_counts()
    .reindex(european_regions)
)


regional_descriptive_frames = []
regional_test_rows = []


for tool, variable in {
    "ROAW": "Mean_ROAW",
    "Lighthouse": "Mean_Lighthouse"
}.items():

    # --------------------------------------------------------
    # Estadísticos descriptivos
    # --------------------------------------------------------

    temp_desc = (
        regional_data
        .groupby("M49 region")[variable]
        .agg(
            n="count",
            Mean="mean",
            SD="std",
            Median="median",
            Min="min",
            Max="max"
        )
        .reindex(european_regions)
        .reset_index()
    )

    temp_desc.insert(0, "Tool", tool)

    regional_descriptive_frames.append(temp_desc)


    # --------------------------------------------------------
    # Kruskal-Wallis
    # --------------------------------------------------------

    groups = [
        regional_data.loc[
            regional_data["M49 region"] == region,
            variable
        ].dropna()
        for region in european_regions
    ]

    H, p = kruskal(*groups)

    n = sum(len(group) for group in groups)
    k = len(european_regions)

    epsilon_squared = (
        (H - k + 1) /
        (n - k)
    )

    regional_test_rows.append({
        "Tool": tool,
        "n": n,
        "H": H,
        "p": p,
        "Epsilon_squared": epsilon_squared
    })


regional_descriptives = pd.concat(
    regional_descriptive_frames,
    ignore_index=True
)

regional_tests = pd.DataFrame(
    regional_test_rows
)


# Versiones redondeadas para visualización
regional_descriptives_display = regional_descriptives.copy()

for col in ["Mean", "SD", "Median", "Min", "Max"]:
    regional_descriptives_display[col] = (
        regional_descriptives_display[col].round(2)
    )


regional_tests_display = regional_tests.copy()

for col in ["H", "p", "Epsilon_squared"]:
    regional_tests_display[col] = (
        regional_tests_display[col].round(4)
    )


print("\nEstadísticos descriptivos por región:")
display(regional_descriptives_display)

print("\nPruebas de Kruskal-Wallis:")
display(regional_tests_display)

Número de países por región:
M49 region
Eastern Europe      6
Northern Europe    10
Southern Europe     7
Western Europe      7
Name: count, dtype: int64

Estadísticos descriptivos por región:


,Tool,M49 region,n,Mean,SD,Median,Min,Max
0,ROAW,Eastern Europe,6,5.99,0.53,5.96,5.15,6.76
1,ROAW,Northern Europe,10,6.62,0.75,6.70,5.65,7.87
2,ROAW,Southern Europe,6,5.82,0.71,5.84,4.67,6.58
3,ROAW,Western Europe,7,6.72,0.67,6.66,5.70,7.74
4,Lighthouse,Eastern Europe,6,84.93,7.25,86.10,73.00,93.20
5,Lighthouse,Northern Europe,10,94.62,4.04,95.80,88.00,99.40
6,Lighthouse,Southern Europe,7,88.26,5.45,90.40,76.60,92.40
7,Lighthouse,Western Europe,7,93.11,4.45,93.60,84.60,97.20



Pruebas de Kruskal-Wallis:


,Tool,n,H,p,Epsilon_squared
0,ROAW,29,6.0107,0.1111,0.1204
1,Lighthouse,30,11.3282,0.0101,0.3203


### 6.1. Comparaciones post hoc entre regiones

Cuando la prueba de Kruskal-Wallis muestra diferencias estadísticamente significativas entre las cuatro regiones, se realizan comparaciones post hoc mediante la prueba de Dunn.

Los valores `p` de las comparaciones por pares se ajustan mediante el procedimiento de Holm para controlar el error asociado a las múltiples comparaciones.

Las comparaciones post hoc únicamente se realizan para las herramientas cuyo contraste global de Kruskal-Wallis presenta un valor `p < .05`.

In [ ]:
!pip -q install scikit-posthocs

In [ ]:
import scikit_posthocs as sp


dunn_results_by_tool = {}


for tool, variable in {
    "ROAW": "Mean_ROAW",
    "Lighthouse": "Mean_Lighthouse"
}.items():

    global_p = regional_tests.loc[
        regional_tests["Tool"] == tool,
        "p"
    ].iloc[0]

    if global_p < 0.05:

        tool_data = regional_data[
            ["M49 region", variable]
        ].dropna()

        dunn_results = sp.posthoc_dunn(
            tool_data,
            val_col=variable,
            group_col="M49 region",
            p_adjust="holm"
        )

        dunn_results = dunn_results.loc[
            european_regions,
            european_regions
        ]

        dunn_results_by_tool[tool] = dunn_results

        print(f"\nDunn post hoc — {tool}:")
        display(dunn_results.round(4))

    else:
        print(
            f"\n{tool}: no se realizan comparaciones post hoc "
            f"(Kruskal-Wallis p = {global_p:.4f})."
        )


ROAW: no se realizan comparaciones post hoc (Kruskal-Wallis p = 0.1111).

Dunn post hoc — Lighthouse:


,Eastern Europe,Northern Europe,Southern Europe,Western Europe
Eastern Europe,1.0000,0.0260,1.0000,0.1045
Northern Europe,0.0260,1.0000,0.1045,1.0000
Southern Europe,1.0000,0.1045,1.0000,0.2345
Western Europe,0.1045,1.0000,0.2345,1.0000


### 6.2. Diferencias regionales por categoría de sitio web

Como análisis exploratorio, se evalúa si existen diferencias regionales dentro de cada una de las cinco categorías de sitios web.

El análisis se realiza por separado para ROAW y Lighthouse mediante pruebas de Kruskal-Wallis.

En total se realizan diez contrastes (cinco categorías por dos herramientas), por lo que los valores `p` se ajustan conjuntamente mediante el procedimiento de Benjamini-Hochberg para controlar la tasa de falsos descubrimientos (FDR).

Estos resultados se interpretan como análisis secundarios y exploratorios.

In [ ]:
# Incorporar la región M49 a los datos a nivel de sitio web
regional_category_data = accessibility.merge(
    country[
        [
            "Country",
            "M49 region"
        ]
    ],
    on="Country",
    how="left",
    validate="many_to_one"
)


regional_category_data = regional_category_data[
    regional_category_data["M49 region"].isin(
        european_regions
    )
].copy()


regional_category_rows = []


for tool, score_col in {
    "ROAW": "Score_ROAW",
    "Lighthouse": "Score_Lighthouse"
}.items():

    for category in category_order:

        groups = [
            regional_category_data.loc[
                (regional_category_data["M49 region"] == region) &
                (regional_category_data["Category"] == category),
                score_col
            ].dropna()
            for region in european_regions
        ]

        H, p = kruskal(*groups)

        n = sum(len(group) for group in groups)

        regional_category_rows.append({
            "Tool": tool,
            "Category": category,
            "n": n,
            "H": H,
            "p": p
        })


regional_category_results = pd.DataFrame(
    regional_category_rows
)


# FDR sobre los 10 contrastes
regional_category_results["q"] = multipletests(
    regional_category_results["p"],
    method="fdr_bh"
)[1]


regional_category_display = (
    regional_category_results.copy()
)

for col in ["H", "p", "q"]:
    regional_category_display[col] = (
        regional_category_display[col].round(4)
    )


display(regional_category_display)


print("\nResultados con q < .05:")
display(
    regional_category_display.loc[
        regional_category_results["q"] < 0.05
    ]
)

,Tool,Category,n,H,p,q
0,ROAW,National gov.,21,1.1367,0.7682,0.8029
1,ROAW,Local gov.,21,2.9676,0.3966,0.5804
2,ROAW,Tourism,27,0.9932,0.8029,0.8029
3,ROAW,Public transport,25,6.9541,0.0734,0.2446
4,ROAW,News portals,28,5.1132,0.1637,0.3291
5,Lighthouse,National gov.,30,1.1910,0.7552,0.8029
6,Lighthouse,Local gov.,30,7.7375,0.0518,0.2446
7,Lighthouse,Tourism,30,8.1830,0.0424,0.2446
8,Lighthouse,Public transport,30,2.9063,0.4063,0.5804
9,Lighthouse,News portals,30,5.1008,0.1646,0.3291



Resultados con q < .05:


,Tool,Category,n,H,p,q


## 7. Asociación entre las puntuaciones de ROAW y Lighthouse

Dado que ROAW y Lighthouse evalúan los mismos sitios web y las evaluaciones válidas de ambas herramientas se realizaron el mismo día, se analiza la relación entre sus puntuaciones.

Las herramientas utilizan sistemas de puntuación y escalas diferentes (0–10 en ROAW y 0–100 en Lighthouse), por lo que no se comparan directamente sus valores absolutos ni se calculan diferencias entre puntuaciones.

El objetivo es evaluar hasta qué punto ambas herramientas producen patrones similares en la valoración relativa de los sitios web. Para ello se utilizan los coeficientes de correlación de Pearson y Spearman sobre las evaluaciones con puntuaciones válidas en ambas herramientas.

### 7.1. Asociación global a nivel de sitio web

Se analiza la asociación entre las puntuaciones de ROAW y Lighthouse utilizando únicamente los sitios web que disponen de una evaluación válida en ambas herramientas.

In [ ]:
# Sitios con puntuación válida en ambas herramientas
paired_scores = accessibility[
    accessibility[
        ["Score_ROAW", "Score_Lighthouse"]
    ].notna().all(axis=1)
].copy()


n_paired = len(paired_scores)

pearson_r, pearson_p = pearsonr(
    paired_scores["Score_ROAW"],
    paired_scores["Score_Lighthouse"]
)

spearman_rho, spearman_p = spearmanr(
    paired_scores["Score_ROAW"],
    paired_scores["Score_Lighthouse"]
)


tool_association_site = pd.DataFrame([{
    "n": n_paired,
    "Pearson_r": pearson_r,
    "Pearson_p": pearson_p,
    "Spearman_rho": spearman_rho,
    "Spearman_p": spearman_p
}])


tool_association_site_display = (
    tool_association_site.copy()
)

for col in [
    "Pearson_r",
    "Pearson_p",
    "Spearman_rho",
    "Spearman_p"
]:
    tool_association_site_display[col] = (
        tool_association_site_display[col].round(4)
    )


display(tool_association_site_display)

,n,Pearson_r,Pearson_p,Spearman_rho,Spearman_p
0,126,0.4091,0.0,0.4464,0.0


### 7.2. Asociación entre herramientas a nivel de país utilizando evaluaciones emparejadas

Para evitar que las diferencias de cobertura entre herramientas influyan en la comparación a nivel nacional, se calculan nuevas puntuaciones medias utilizando únicamente las categorías que disponen simultáneamente de una puntuación válida de ROAW y Lighthouse.

Se exige un mínimo de tres categorías emparejadas por país, en consonancia con el criterio utilizado para las puntuaciones medias nacionales.

De este modo, las dos medias comparadas para cada país se basan exactamente en el mismo conjunto de sitios web.

In [ ]:
paired_country = (
    paired_scores
    .groupby("Country")
    .agg(
        n_paired_categories=("Category", "count"),
        Mean_ROAW_paired=("Score_ROAW", "mean"),
        Mean_Lighthouse_paired=("Score_Lighthouse", "mean")
    )
    .reset_index()
)


# Exigir al menos tres categorías emparejadas
paired_country_valid = paired_country[
    paired_country["n_paired_categories"] >= 3
].copy()


print("Cobertura de categorías emparejadas por país:")
print(
    paired_country["n_paired_categories"]
    .value_counts()
    .sort_index()
)


print(
    "\nPaíses incluidos en el análisis:",
    len(paired_country_valid)
)


pearson_r, pearson_p = pearsonr(
    paired_country_valid["Mean_ROAW_paired"],
    paired_country_valid["Mean_Lighthouse_paired"]
)

spearman_rho, spearman_p = spearmanr(
    paired_country_valid["Mean_ROAW_paired"],
    paired_country_valid["Mean_Lighthouse_paired"]
)


tool_association_country = pd.DataFrame([{
    "n": len(paired_country_valid),
    "Pearson_r": pearson_r,
    "Pearson_p": pearson_p,
    "Spearman_rho": spearman_rho,
    "Spearman_p": spearman_p
}])


tool_association_country_display = (
    tool_association_country.copy()
)

for col in [
    "Pearson_r",
    "Pearson_p",
    "Spearman_rho",
    "Spearman_p"
]:
    tool_association_country_display[col] = (
        tool_association_country_display[col].round(4)
    )


display(tool_association_country_display)

Cobertura de categorías emparejadas por país:
n_paired_categories
2     1
3     9
4     8
5    13
Name: count, dtype: int64

Países incluidos en el análisis: 30


,n,Pearson_r,Pearson_p,Spearman_rho,Spearman_p
0,30,0.5559,0.0014,0.5135,0.0037


## 8. Análisis de sensibilidad

En los análisis principales, la puntuación media nacional de cada herramienta se considera válida cuando existen resultados para al menos tres de las cinco categorías de sitios web.

Para evaluar si este criterio de disponibilidad influye en los resultados, se repiten las asociaciones principales utilizando únicamente países con puntuaciones válidas en las cinco categorías de cada herramienta.

Este análisis es especialmente restrictivo para ROAW, debido a la mayor frecuencia de evaluaciones sin resultado. Por ello, la sensibilidad se interpreta atendiendo principalmente a la dirección y magnitud de las asociaciones, además de a su significación estadística.

No se realiza ninguna imputación de valores ausentes.

### 8.1. Correlaciones nacionales con casos completos

Se repiten las correlaciones entre accesibilidad y HDI, SPI, EGDI y acceso a Internet utilizando únicamente los países con resultados válidos en las cinco categorías de cada herramienta.

Al igual que en el análisis principal, se calculan los coeficientes de Pearson y Spearman y se aplica una corrección de Benjamini-Hochberg de forma independiente a los ocho contrastes de cada método.

In [ ]:
sensitivity_rows = []

for tool, score_col, coverage_col in [
    ("ROAW", "Mean_ROAW", "n_categories_ROAW"),
    ("Lighthouse", "Mean_Lighthouse", "n_categories_Lighthouse")
]:

    # Solo países con las cinco categorías disponibles
    complete_data = analysis_data[
        analysis_data[coverage_col] == 5
    ].copy()

    for indicator in indicators:

        data = complete_data[
            [score_col, indicator]
        ].dropna()

        n = len(data)

        pearson_r, pearson_p = pearsonr(
            data[score_col],
            data[indicator]
        )

        spearman_rho, spearman_p = spearmanr(
            data[score_col],
            data[indicator]
        )

        sensitivity_rows.append({
            "Tool": tool,
            "Indicator": indicator,
            "n": n,
            "Pearson_r": pearson_r,
            "Pearson_p": pearson_p,
            "Spearman_rho": spearman_rho,
            "Spearman_p": spearman_p
        })


sensitivity_correlations = pd.DataFrame(
    sensitivity_rows
)


# Corrección FDR
sensitivity_correlations["Pearson_q"] = multipletests(
    sensitivity_correlations["Pearson_p"],
    method="fdr_bh"
)[1]

sensitivity_correlations["Spearman_q"] = multipletests(
    sensitivity_correlations["Spearman_p"],
    method="fdr_bh"
)[1]


# Versión redondeada para visualización
sensitivity_correlations_display = (
    sensitivity_correlations.copy()
)

for col in [
    "Pearson_r",
    "Pearson_p",
    "Pearson_q",
    "Spearman_rho",
    "Spearman_p",
    "Spearman_q"
]:
    sensitivity_correlations_display[col] = (
        sensitivity_correlations_display[col].round(4)
    )


display(sensitivity_correlations_display)

,Tool,Indicator,n,Pearson_r,Pearson_p,Spearman_rho,Spearman_p,Pearson_q,Spearman_q
0,ROAW,HDI,14,0.5991,0.0236,0.5771,0.0307,0.0377,0.0491
1,ROAW,SPI,14,0.5447,0.0440,0.5039,0.0662,0.0503,0.0757
2,ROAW,EGDI,13,0.4604,0.1134,0.3631,0.2226,0.1134,0.2226
3,ROAW,Internet access,14,0.5620,0.0365,0.5611,0.0369,0.0486,0.0491
4,Lighthouse,HDI,30,0.7473,0.0000,0.6845,0.0000,0.0000,0.0001
5,Lighthouse,SPI,29,0.8007,0.0000,0.7293,0.0000,0.0000,0.0001
6,Lighthouse,EGDI,30,0.6032,0.0004,0.5608,0.0013,0.0011,0.0034
7,Lighthouse,Internet access,30,0.4783,0.0075,0.4128,0.0234,0.0150,0.0468


## 9. Preparación de tablas para el manuscrito

A partir de los resultados obtenidos se generan las tablas utilizadas en el manuscrito. Las tablas mantienen las escalas originales de ROAW y Lighthouse y presentan únicamente los análisis seleccionados para el cuerpo principal del artículo.

### 9.1. Estadísticos descriptivos por categoría y herramienta

Se preparan los estadísticos descriptivos de las cinco categorías de sitios web para ROAW y Lighthouse. Para cada combinación se presentan el número de observaciones válidas, media, desviación estándar, mediana, mínimo y máximo.

In [ ]:
def descriptive_by_category(df, tool_name, score_col="Score"):
    table = (
        df.groupby("Category")[score_col]
        .agg(
            n="count",
            Mean="mean",
            SD="std",
            Median="median",
            Min="min",
            Max="max"
        )
        .reindex(category_order)
        .reset_index()
    )

    table.insert(0, "Tool", tool_name)

    return table


table2_roaw = descriptive_by_category(
    roaw,
    "ROAW"
)

table2_lighthouse = descriptive_by_category(
    lighthouse,
    "Lighthouse"
)

table2 = pd.concat(
    [table2_roaw, table2_lighthouse],
    ignore_index=True
)


# Redondeo para presentación
table2_display = table2.copy()

for col in ["Mean", "SD", "Median", "Min", "Max"]:
    table2_display[col] = table2_display[col].round(2)

display(table2_display)

,Tool,Category,n,Mean,SD,Median,Min,Max
0,ROAW,National gov.,22,7.23,1.32,6.80,5.3,9.5
1,ROAW,Local gov.,22,6.59,1.46,6.50,3.5,9.5
2,ROAW,Tourism,28,6.09,1.22,5.80,3.7,9.4
3,ROAW,Public transport,26,5.86,1.54,6.05,3.2,8.4
4,ROAW,News portals,29,6.06,1.08,6.10,3.9,8.4
5,Lighthouse,National gov.,31,92.23,13.93,96.00,34.0,100.0
6,Lighthouse,Local gov.,31,92.29,10.32,97.00,63.0,100.0
7,Lighthouse,Tourism,31,89.97,8.45,92.00,73.0,100.0
8,Lighthouse,Public transport,30,90.17,8.89,92.50,69.0,100.0
9,Lighthouse,News portals,31,88.77,9.14,90.00,65.0,100.0


### 9.2. Asociación entre accesibilidad e indicadores nacionales

Se prepara la tabla de asociaciones entre las puntuaciones medias nacionales de ROAW y Lighthouse y los cuatro indicadores nacionales. Se presentan el tamaño muestral y los coeficientes de Pearson y Spearman, junto con los valores p sin ajustar y los valores q corregidos mediante Benjamini-Hochberg.

In [ ]:
from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests

table3_rows = []

for tool, score_col in [
    ("ROAW", "Mean_ROAW"),
    ("Lighthouse", "Mean_Lighthouse")
]:
    for indicator in indicators:

        data = analysis_data[
            [score_col, indicator]
        ].dropna()

        pearson_r, pearson_p = pearsonr(
            data[score_col],
            data[indicator]
        )

        spearman_rho, spearman_p = spearmanr(
            data[score_col],
            data[indicator]
        )

        table3_rows.append({
            "Tool": tool,
            "Indicator": indicator,
            "n": len(data),
            "Pearson_r": pearson_r,
            "Pearson_p": pearson_p,
            "Spearman_rho": spearman_rho,
            "Spearman_p": spearman_p
        })


table3_raw = pd.DataFrame(table3_rows)


# Corrección FDR conjunta para los 8 contrastes de cada método
table3_raw["Pearson_q"] = multipletests(
    table3_raw["Pearson_p"],
    method="fdr_bh"
)[1]

table3_raw["Spearman_q"] = multipletests(
    table3_raw["Spearman_p"],
    method="fdr_bh"
)[1]

In [ ]:
def format_p(value):
    if value < 0.001:
        return "<.001"
    return f"={value:.3f}".replace("0.", ".")


def format_corr(coef, p, q):
    return (
        f"{coef:.3f} "
        f"(p{format_p(p)}; q{format_p(q)})"
    )


table3 = table3_raw.copy()

table3["Pearson r (p; q)"] = table3.apply(
    lambda row: format_corr(
        row["Pearson_r"],
        row["Pearson_p"],
        row["Pearson_q"]
    ),
    axis=1
)

table3["Spearman ρ (p; q)"] = table3.apply(
    lambda row: format_corr(
        row["Spearman_rho"],
        row["Spearman_p"],
        row["Spearman_q"]
    ),
    axis=1
)


# Nombre utilizado en el manuscrito
table3["Indicator"] = table3["Indicator"].replace({
    "Internet access": "Internet use",
    "Individuals using the Internet": "Internet use"
})


table3_display = table3[
    [
        "Tool",
        "Indicator",
        "n",
        "Pearson r (p; q)",
        "Spearman ρ (p; q)"
    ]
]

display(table3_display)

,Tool,Indicator,n,Pearson r (p; q),Spearman ρ (p; q)
0,ROAW,HDI,30,0.526 (p=.003; q=.005),0.509 (p=.004; q=.007)
1,ROAW,SPI,29,0.544 (p=.002; q=.005),0.544 (p=.002; q=.005)
2,ROAW,EGDI,29,0.469 (p=.010; q=.012),0.403 (p=.030; q=.035)
3,ROAW,Internet use,30,0.412 (p=.024; q=.024),0.356 (p=.054; q=.054)
4,Lighthouse,HDI,31,0.742 (p<.001; q<.001),0.685 (p<.001; q<.001)
5,Lighthouse,SPI,30,0.803 (p<.001; q<.001),0.742 (p<.001; q<.001)
6,Lighthouse,EGDI,30,0.603 (p<.001; q=.001),0.561 (p=.001; q=.003)
7,Lighthouse,Internet use,31,0.489 (p=.005; q=.007),0.423 (p=.018; q=.024)


### 9.3. Diferencias regionales en accesibilidad

Se preparan los estadísticos descriptivos de las puntuaciones medias nacionales por subregión europea y herramienta, junto con los resultados de las pruebas de Kruskal-Wallis y sus tamaños del efecto.

In [ ]:
regional_data = analysis_data[
    analysis_data["M49 region"].isin([
        "Eastern Europe",
        "Northern Europe",
        "Southern Europe",
        "Western Europe"
    ])
].copy()


region_order = [
    "Eastern Europe",
    "Northern Europe",
    "Southern Europe",
    "Western Europe"
]


table4_descriptive_rows = []

for tool, score_col in [
    ("ROAW", "Mean_ROAW"),
    ("Lighthouse", "Mean_Lighthouse")
]:

    for region in region_order:

        values = regional_data.loc[
            regional_data["M49 region"] == region,
            score_col
        ].dropna()

        table4_descriptive_rows.append({
            "Tool": tool,
            "M49 region": region,
            "n": len(values),
            "Mean": values.mean(),
            "SD": values.std(),
            "Median": values.median()
        })


table4_panel_a = pd.DataFrame(
    table4_descriptive_rows
)

table4_panel_a_display = table4_panel_a.copy()

for col in ["Mean", "SD", "Median"]:
    table4_panel_a_display[col] = (
        table4_panel_a_display[col].round(2)
    )

display(table4_panel_a_display)

,Tool,M49 region,n,Mean,SD,Median
0,ROAW,Eastern Europe,6,5.99,0.53,5.96
1,ROAW,Northern Europe,10,6.62,0.75,6.70
2,ROAW,Southern Europe,6,5.82,0.71,5.84
3,ROAW,Western Europe,7,6.72,0.67,6.66
4,Lighthouse,Eastern Europe,6,84.93,7.25,86.10
5,Lighthouse,Northern Europe,10,94.62,4.04,95.80
6,Lighthouse,Southern Europe,7,88.26,5.45,90.40
7,Lighthouse,Western Europe,7,93.11,4.45,93.60


In [ ]:
from scipy.stats import kruskal

table4_test_rows = []

for tool, score_col in [
    ("ROAW", "Mean_ROAW"),
    ("Lighthouse", "Mean_Lighthouse")
]:

    groups = []

    for region in region_order:
        values = regional_data.loc[
            regional_data["M49 region"] == region,
            score_col
        ].dropna()

        groups.append(values)

    H, p = kruskal(*groups)

    n_total = sum(len(group) for group in groups)
    k = len(groups)

    epsilon_squared = (
        (H - k + 1) / (n_total - k)
    )

    table4_test_rows.append({
        "Tool": tool,
        "n": n_total,
        "H": H,
        "p": p,
        "Epsilon_squared": epsilon_squared
    })


table4_panel_b = pd.DataFrame(
    table4_test_rows
)


table4_panel_b_display = table4_panel_b.copy()

for col in ["H", "p", "Epsilon_squared"]:
    table4_panel_b_display[col] = (
        table4_panel_b_display[col].round(3)
    )

display(table4_panel_b_display)

,Tool,n,H,p,Epsilon_squared
0,ROAW,29,6.011,0.111,0.12
1,Lighthouse,30,11.328,0.010,0.32


### 9.4. Exportación de las tablas del manuscrito

Las tablas principales se exportan a un único archivo Excel para facilitar su incorporación al manuscrito y su conservación junto con los resultados reproducibles del estudio.

In [ ]:
table1 = country[
    [
        "Country",
        "Regulatory group",
        "M49 region"
    ]
].copy()

In [ ]:
output_file = "UAIS_final_tables.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    # Tabla 1
    table1.to_excel(
        writer,
        sheet_name="Table 1 - Countries",
        index=False
    )

    # Tabla 2
    table2_display.to_excel(
        writer,
        sheet_name="Table 2 - Descriptives",
        index=False
    )

    # Tabla 3
    table3_display.to_excel(
        writer,
        sheet_name="Table 3 - Correlations",
        index=False
    )

    # Tabla 4 - Panel A
    table4_panel_a_display.to_excel(
        writer,
        sheet_name="Table 4A - Regions",
        index=False
    )

    # Tabla 4 - Panel B
    table4_panel_b_display.to_excel(
        writer,
        sheet_name="Table 4B - Kruskal",
        index=False
    )

print(f"Archivo generado: {output_file}")

Archivo generado: UAIS_final_tables.xlsx


In [ ]:
from google.colab import files

files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>